# Cascade Denoised Smoothing on Kaggle (CIFAR-10)
Settings: **Accelerator = GPU T4 x2**, **Internet = On**, Persistence = *Files only* (if available).

Each phase can be run in its own session/commit. Outputs go to `/kaggle/working/dds/out`.
To resume after a session ends: add this notebook's previous **Output** as an input and set `PREV_OUT` below.

**Save & Run All (Commit)**: set `PHASES` in the config cell first. Typical commits:
1. `["probe", "record", "analyze"]` with `MAX_IMAGES=500` (record stops after `TIME_BUDGET` h).
2. If not finished: add this notebook's Output as input, set `PREV_OUT`, `PHASES=["record","analyze"]`, commit again.

In [ ]:
# ---------------- config ----------------
SIGMA      = 0.25
SKIP       = 20          # certify images idx % 20 == 0 (500 imgs); probe uses the other 9,500
MAX_IMAGES = 100         # start with 100, later raise to 500 (runs resume, nothing is recomputed)
N          = 10000
BATCH      = 500         # lower to 250 on OOM
AMP        = True        # fp16 on the expensive path; checked in cell 3; SAME flag for dds & cascade
DELTA, CAL_ALPHA = 0.01, 0.05
# Which steps to run. "Save & Run All (Commit)" runs every cell, so only listed phases do work:
#   sanity | probe | record | analyze | profile | candidates | direct (old per-mode certify, slow)
PHASES      = ["probe", "record", "analyze"]
TIME_BUDGET = 10.5       # hours for record() per commit; Kaggle kills sessions at 12 h
PREV_OUT   = None        # e.g. "/kaggle/input/<this-notebook-output>/dds/out" to resume
REPO       = "https://github.com/ducquan19/dds.git"
GIT_REF    = "main"      # pin a commit sha for a whole experiment (e.g. "a1b2c3d") so resumed runs use identical code
COHEN_GDRIVE_ID = "1h_TpbXm5haY5f-l4--IKylmdz6tvPoR4"   # models.tar from locuslab/smoothing README

import os, subprocess, torch
SIG = f"{SIGMA:.2f}"
AMPF = "--amp" if AMP else ""
NGPU = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(NGPU)])

In [ ]:
# ---------------- 1. setup (~5 min) ----------------
%cd /kaggle/working
if not os.path.exists("dds"):
    !git clone -q {REPO} dds
%cd /kaggle/working/dds
!git fetch -q origin && git checkout -q {GIT_REF} && (git pull -q origin {GIT_REF} 2>/dev/null || true)
SHA = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("code version:", SHA)
!pip install -q gdown statsmodels "transformers>=4.40,<5"

if not os.path.exists("cifar10/cifar10_uncond_50M_500K.pt"):
    !wget -q -O cifar10/cifar10_uncond_50M_500K.pt https://openaipublic.blob.core.windows.net/diffusion/march-2021/cifar10_uncond_50M_500K.pt
if not os.path.exists("cifar10/archs"):
    !git clone -q --depth 1 https://github.com/locuslab/smoothing.git /tmp/cohen && cp -r /tmp/cohen/code/archs cifar10/archs
if not os.path.exists("models/cifar10/resnet110/noise_0.25/checkpoint.pth.tar"):
    !gdown -q {COHEN_GDRIVE_ID} -O /tmp/models.tar && mkdir -p /tmp/cm && tar -xf /tmp/models.tar -C /tmp/cm && rm /tmp/models.tar
    !mkdir -p models && cp -r "$(dirname "$(dirname "$(find /tmp/cm -type d -path '*cifar10/resnet110/noise_0.25' | head -1)")")" models/cifar10 && rm -rf /tmp/cm
!ls models/cifar10/resnet110
os.makedirs("out", exist_ok=True); os.makedirs("logs", exist_ok=True)
prev_sha = open("out/git_sha.txt").read().strip() if os.path.exists("out/git_sha.txt") else None
if PREV_OUT:
    !cp -rn {PREV_OUT}/. out/ ; ls out
    prev_sha = open("out/git_sha.txt").read().strip() if os.path.exists("out/git_sha.txt") else None
if prev_sha and prev_sha != SHA:
    print(f"WARNING: out/ was produced with code {prev_sha}, now running {SHA}. Set GIT_REF = '{prev_sha}' to resume consistently.")
open("out/git_sha.txt", "w").write(SHA)
import sys; sys.path.insert(0, "cifar10")
from torchvision import datasets
datasets.CIFAR10("data/cifar10", train=False, download=True)
CKPT = f"models/cifar10/resnet110/noise_{SIG}/checkpoint.pth.tar"

In [ ]:
# ---------------- 2. sanity: imports, checkpoint, AMP vs FP32 agreement (~3 min) ----------------
if "sanity" in PHASES:
    import core                      # fails if the binom_test fix is missing
    from DRM_cascade import CascadeRobustModel
    m = CascadeRobustModel(SIGMA, mode="cascade", tau=1.01, cheap_ckpt=CKPT)
    ds = datasets.CIFAR10("data/cifar10", train=False, transform=__import__("torchvision").transforms.ToTensor())
    x = torch.stack([ds[i][0] for i in range(1, 200, 2)]).cuda().repeat_interleave(5, 0)   # not certify images
    torch.manual_seed(0); xn = x + torch.randn_like(x) * SIGMA
    m.amp = False; p32 = torch.cat([m.expensive_logits(c).argmax(1) for c in xn.split(250)])
    m.amp = True;  p16 = torch.cat([m.expensive_logits(c).argmax(1) for c in xn.split(250)])
    y = torch.tensor([ds[i][1] for i in range(1, 200, 2)]).cuda().repeat_interleave(5)
    print(f"AMP/FP32 agreement {(p32 == p16).float().mean():.4f} | dds acc fp32 {(p32 == y).float().mean():.3f} amp {(p16 == y).float().mean():.3f}")
    print(f"cheap acc {(m.cheap_logits(xn).argmax(1) == y).float().mean():.3f}")
    # timing of one certify-sized pass (N0+N samples) on one GPU
    import time
    for amp in [False, True]:
        m.amp = amp; torch.cuda.synchronize(); t0 = time.time()
        for _ in range((N + 100) // BATCH): m.expensive_logits(x[:1].repeat(BATCH, 1, 1, 1) + torch.randn(BATCH, 3, 32, 32, device="cuda") * SIGMA)
        torch.cuda.synchronize(); print(f"amp={amp}: dds ~{time.time() - t0:.1f} s/img on 1 GPU")
    del m; torch.cuda.empty_cache()
    # -> if agreement < 0.99, set AMP = False in the config cell

In [ ]:
# helper: run certify_cascade on all GPUs in parallel (one shard per GPU), resumable
def certify(mode, tau=None, tag=None):
    tag = tag or mode
    procs = []
    for s in range(NGPU):
        out = f"out/{tag}_{SIG}_s{s}.tsv"
        cmd = (f"CUDA_VISIBLE_DEVICES={s} PYTHONPATH=cifar10 python cifar10/certify_cascade.py --mode {mode} "
               f"--sigma {SIGMA} --cheap_ckpt {CKPT} --skip {SKIP} --max_images {MAX_IMAGES} --N {N} "
               f"--alpha 0.0005 --batch_size {BATCH} --shard {s} --num_shards {NGPU} --resume "
               f"--outfile {out} {AMPF} {'' if tau is None else '--tau ' + str(tau)} "
               f"> logs/{tag}_{SIG}_s{s}.log 2>&1")
        procs.append(subprocess.Popen(cmd, shell=True))
    rc = [p.wait() for p in procs]
    !tail -n 2 logs/{tag}_{SIG}_s*.log
    assert all(r == 0 for r in rc), f"{tag} failed, see logs/"

def report(*extra):
    files = [f"{t}=out/{t}_{SIG}_s*.tsv" for t in ["dds", "cheap"] + list(extra)]
    files = [f for f in files if __import__("glob").glob(f.split("=")[1])]
    fs = " ".join(files)
    !PYTHONPATH=cifar10 python cifar10/compare.py {fs} --intersect

In [ ]:
# ---------------- 3. probe + calibrate tau  (feasibility test, ~15-30 min) ----------------
if "probe" in PHASES:
    !PYTHONPATH=cifar10 python cifar10/probe.py --sigma {SIGMA} --cheap_ckpt {CKPT} --num_images 9500 --K 4 \
        --split test_holdout --certify_skip {SKIP} --batch_size {BATCH} {AMPF} --out out/probe_{SIG}.npz
    !PYTHONPATH=cifar10 python cifar10/calibrate_tau.py out/probe_{SIG}.npz --delta {DELTA} --alpha {CAL_ALPHA} --out out/tau_{SIG}.txt
    TAU = float(open(f"out/tau_{SIG}.txt").read())
    print("calibrated tau =", TAU)
    # DECISION: accept rate at this tau ~ fraction of diffusion calls saved. < ~0.5  => speed-up < 2x.

In [ ]:
# ---------------- 4. certify: cascade (calibrated tau) + cheap ----------------
if "direct" in PHASES:
    TAU = float(open(f"out/tau_{SIG}.txt").read())
    certify("cascade", tau=TAU, tag="cascade_cal")
    certify("cheap")
    report("cascade_cal")

In [ ]:
# ---------------- 5. certify: dds baseline (most expensive; resumable) ----------------
if "direct" in PHASES:
    certify("dds")
    report("cascade_cal")

In [ ]:
# ---------------- 6. tau sweep for the Pareto plot (optional, run when budget allows) ----------------
if "direct" in PHASES:
    for t in [0.5, 0.7, 0.9, 0.99]:
        certify("cascade", tau=t, tag=f"cascade_tau{t}")
    report("cascade_cal", *[f"cascade_tau{t}" for t in [0.5, 0.7, 0.9, 0.99]])

## Phase 2 — raise the speed-up (after the 100-image run)
Cost model observed: `sec/img(cascade) ≈ cheap + exp_frac × dds`. Two levers: make `cheap` ~free, and lower `exp_frac`
with a stronger noise-trained cheap model. Each candidate only needs **probe + calibrate** (~30 min), no certify.

In [ ]:
# ---------------- 9. profile: where does the time go? (~2 min) ----------------
if "profile" in PHASES:
    from DRM_cascade import CascadeRobustModel
    for amp in [False, True]:
        m = CascadeRobustModel(SIGMA, mode="cascade", tau=1.01, cheap_ckpt=CKPT, amp=amp)
        r = m.profile(batch=BATCH)
        print(f"amp={amp}: sec per 10k samples (1 GPU) ->", {k: round(v, 2) for k, v in r.items()})
        del m; torch.cuda.empty_cache()
    # If vit >> unet, the bottleneck of one-step DDS on CIFAR is the ViT at 224px, not the diffusion model.

In [ ]:
# ---------------- 10. compare cheap models by accept rate at delta (probe + calibrate only) ----------------
if "candidates" in PHASES:
    # Download each checkpoint (Cohen format, CIFAR ResNet-110, trained at THIS sigma) and/or upload as a Kaggle Dataset.
    CANDIDATES = {
        "cohen":       CKPT,
        # "smoothadv":   "/kaggle/input/<dataset>/smoothadv_resnet110_0.25.pth.tar",
        # "consistency": "/kaggle/input/<dataset>/consistency_resnet110_0.25.pth.tar",
    }
    import numpy as np
    for name, ck in CANDIDATES.items():
        out = f"out/probe_{SIG}_{name}_amp{int(AMP)}.npz"
        if not os.path.exists(out):
            !PYTHONPATH=cifar10 python cifar10/probe.py --sigma {SIGMA} --cheap_ckpt {ck} --num_images 9500 --K 4 \
                --split test_holdout --certify_skip {SKIP} --batch_size {BATCH} {AMPF} --out {out} > logs/probe_{name}.log 2>&1
        print(f"==== {name}")
        !tail -n 7 logs/probe_{name}.log | head -n 1
        for d in [0.005, 0.01, 0.02]:
            !PYTHONPATH=cifar10 python cifar10/calibrate_tau.py {out} --delta {d} --alpha {CAL_ALPHA} | sed -n 2p | sed "s/^/delta={d}  /"
    # Pick the candidate with the highest accept rate at delta=0.01, write its tau, then rerun cell 6 with it:
    #   CKPT = CANDIDATES["<best>"]; and calibrate_tau.py ... --out out/tau_{SIG}.txt

## FAST PATH — record once, analyse every mode (recommended)
One GPU pass stores cheap + expensive predictions for all N0+N noise samples of each image
(~(t_cheap + t_exp) per image ≈ 55 s on one T4 with AMP → 500 images ≈ 3.7 h on T4 x2).
Then dds / cheap / cascade at ANY tau are certified on CPU in seconds. Replaces cells 6–8.
Needs: cells 1–2 (setup), and cell 5 (probe → `out/tau_<sigma>.txt`) run with the same `AMP` flag.

In [ ]:
# ---------------- R1. record (resumable: rerun after a session ends, finished images are skipped) ----------------
def record(ckpt=None, out_dir=None, only_cheap=False):
    ckpt = ckpt or CKPT; out_dir = out_dir or f"out/rec_{SIG}"
    procs = []
    for s in range(NGPU):
        cmd = (f"CUDA_VISIBLE_DEVICES={s} PYTHONPATH=cifar10 python cifar10/record.py --sigma {SIGMA} "
               f"--cheap_ckpt {ckpt} --skip {SKIP} --max_images {MAX_IMAGES} --N {N} --batch_size {BATCH} "
               f"--shard {s} --num_shards {NGPU} {AMPF} --time_budget {TIME_BUDGET} {'--only_cheap' if only_cheap else ''} "
               f"--out_dir {out_dir} > logs/record_{os.path.basename(out_dir)}_s{s}.log 2>&1")
        procs.append(subprocess.Popen(cmd, shell=True))
    rc = [p.wait() for p in procs]
    lg = f"logs/record_{os.path.basename(out_dir)}_s*.log"
    !tail -n 2 {lg}
    assert all(r == 0 for r in rc), "record failed, see logs/"

if "record" in PHASES:
    record()

In [ ]:
# ---------------- R2. analyse (CPU, seconds) ----------------
if "analyze" in PHASES:
    !PYTHONPATH=cifar10 python cifar10/analyze_record.py out/rec_{SIG} --tau_file out/tau_{SIG}.txt --paired --tsv_dir out/tsv

In [ ]:
# ---------------- R3. new cheap model on the SAME noise: only the cheap path is run (~t_cheap per image) ----------------
# NEW_CKPT = "/kaggle/input/<dataset>/smoothadv_resnet110_0.25.pth.tar"; NAME = "smoothadv"
# 1) calibrate its tau on held-out images (cell 10 or):
# !PYTHONPATH=cifar10 python cifar10/probe.py --sigma {SIGMA} --cheap_ckpt {NEW_CKPT} --num_images 9500 --K 4 --split test_holdout --certify_skip {SKIP} --batch_size {BATCH} {AMPF} --out out/probe_{SIG}_{NAME}.npz
# !PYTHONPATH=cifar10 python cifar10/calibrate_tau.py out/probe_{SIG}_{NAME}.npz --delta {DELTA} --alpha {CAL_ALPHA} --out out/tau_{SIG}_{NAME}.txt
# 2) record its cheap path and analyse against the stored expensive predictions:
# record(NEW_CKPT, f"out/rec_{SIG}_{NAME}", only_cheap=True)
# !PYTHONPATH=cifar10 python cifar10/analyze_record.py out/rec_{SIG} --cheap_rec out/rec_{SIG}_{NAME} --tau_file out/tau_{SIG}_{NAME}.txt --paired